In [1]:
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever


raw_documents = [
    Document(page_content="보안 규정: 업무용 PC는 3개월마다 비밀번호를 변경해야 하며 특수문자를 포함해야 합니다."),
    Document(page_content="원격 근무 가이드: 재택 근무 시 반드시 사내 VPN을 연결하여 사내 인트라넷에 접속해야 합니다."),
    Document(page_content="장비 지급 규정: 신규 입사자에게는 노트북 1대와 외장 모니터 1대가 기본 지급됩니다."),
    Document(page_content="계정 관리: 사내 계정 5회 연속 로그인 실패 시 계정이 잠기며 IT 지원팀에 잠금 해제를 요청해야 합니다.")
]



In [2]:
def custom_tokenizer(text):
    return text.split()

bm25_retriever = BM25Retriever.from_documents(
    documents=raw_documents,
    preprocess_func=custom_tokenizer
)

bm25_retriever.k = 2


In [4]:
import os

from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_postgres import PGVector

# chapter05/.env 파일의 환경변수를 불러옵니다.
load_dotenv()

connection_url = os.getenv("DATABASE_URL")

# 비밀값을 출력하지 않고 설정 여부만 검사합니다.
if not connection_url:
    raise ValueError(
        "DATABASE_URL을 찾지 못했습니다. chapter05/.env 파일을 확인하세요."
    )

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vectorstore = PGVector.from_documents(
    documents=raw_documents,
    embedding=embeddings,
    connection=connection_url,
    collection_name="it_regulations_collection",
    use_jsonb=True,
)

vector_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

In [ ]:
# EnsembleRetriever 결합 및 Hybrid 검색 실행

from langchain_classic.retrievers import EnsembleRetriever

# BM25 검색기와 PostgreSQL 벡터 검색기를 5:5 비중으로 결합합니다.
hybrid_retriever = EnsembleRetriever(
    # 결합할 검색기 객체를 리스트로 등록합니다.
    retrievers=[bm25_retriever, vector_retriever],
    # 두 검색기 결과의 반영 비율을 균등하게 설정합니다.
    weights=[0.5, 0.5] # 가중치 부여 / 일반적으로 LLM으로 가중치의 비중 여부를 파악해서 가중치를 조정 // 직접 조정하는 건 X
)

# 키워드(VPN)와 뉘앙스(외부 접속)가 섞인 검색 질의를 작성합니다.
query = "외부 장소에서 회사 시스템 연결 시 VPN 보안 수칙"

# Hybrid 검색기를 실행하여 최적의 관련 문서를 추출합니다.
retrieved_docs = hybrid_retriever.invoke(query)

print(f"입력 검색어: {query}\n")
print("[Hybrid 검색 결과]")
for idx, doc in enumerate(retrieved_docs):
    print(f"{idx + 1} 순위: {doc.page_content}")

입력 검색어: 외부 장소에서 회사 시스템 연결 시 VPN 보안 수칙

[Hybrid 검색 결과]
1 순위: 보안 규정: 업무용 PC는 3개월마다 비밀번호를 변경해야 하며 특수문자를 포함해야 합니다.
2 순위: 원격 근무 가이드: 재택 근무 시 반드시 사내 VPN을 연결하여 사내 인트라넷에 접속해야 합니다.
3 순위: 계정 관리: 사내 계정 5회 연속 로그인 실패 시 계정이 잠기며 IT 지원팀에 잠금 해제를 요청해야 합니다.


In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate

model = init_chat_model(
    model="gpt-5-mini",
    model_provider="openai",
    temperature=0
)

# 검색된 문맥을 반영할 프롬프트 구조를 설계합니다.
qa_prompt = PromptTemplate.from_template(
    "다음 제공된 [참고 문서]를 바탕으로 [질문]에 대해 간결하고 정확하게 답변하세요.\n\n"
    "[참고 문서]:\n{context}\n\n"
    "[질문]:\n{question}\n\n"
    "[답변]:"
)

# 결합 검색 결과 문서들을 줄바꿈 문자로 연결합니다.
context_text = "\n".join([doc.page_content for doc in retrieved_docs])

# 프롬프트와 LLM 모델을 파이프 연산자로 연결합니다.
qa_chain = qa_prompt | model

# 질의응답 체인을 실행합니다.
response = qa_chain.invoke({
    "context": context_text,
    "question": query
})

print("[최종 답변]")
print(response.content)